# RQ3 Deterministic Report Evaluation

This notebook evaluates report completion, NCT citation coverage, identifier-level grounding, citation status, and output-format compliance. It is independent of the LLM-as-a-judge evaluation and can be rerun while the final report runs are still growing.

In [ ]:
from pathlib import Path
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import deterministic_report_metrics as report_metrics

# Reload local metric definitions when this notebook is rerun after edits.
report_metrics = importlib.reload(report_metrics)
build_report_metrics = report_metrics.build_report_metrics
citation_status_summary = report_metrics.citation_status_summary
export_tables = report_metrics.export_tables
macro_summary = report_metrics.macro_summary
micro_summary = report_metrics.micro_summary

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

## Configuration

Only the final v2 report runs are included. The 51 high-confidence reviews are assembled from the two index shards and the development-completion run for each model.

In [ ]:
FINAL_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'gpt_researcher').is_dir()),
    None,
)
if FINAL_ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the repository.')
NOTEBOOK_DIR = FINAL_ROOT / 'evaluation' / 'reports'
# RUN_ROOT = FINAL_ROOT / 'MARIA' / 'outputs' / 'outputs'
RUN_ROOT = FINAL_ROOT / 'outputs'


GROUND_TRUTH_PATH = FINAL_ROOT / 'benchmark' / 'benchmark.json'
DEV_PMIDS_PATH = FINAL_ROOT / 'benchmark' / 'benchmark.json'
RESULTS_DIR = FINAL_ROOT / 'outputs' / 'evaluation' / 'deterministic'
FIGURES_DIR = RESULTS_DIR / 'figures'

RUN_DIRECTORIES = {
    'Qwen3-8B': [
        RUN_ROOT / 'final_qwen_b4d2_v2_idx1_440',
        RUN_ROOT / 'final_qwen_b4d2_v2_idx441_881',
        RUN_ROOT / 'final_qwen_b4d2_v2_development_completion',
    ],
    'Fleming-R1-7B': [
        RUN_ROOT / 'final_fleming_b4d2_v2_idx1_440',
        RUN_ROOT / 'final_fleming_b4d2_v2_idx441_881',
        RUN_ROOT / 'final_fleming_b4d2_v2_development_completion',
    ],
}

for model, directories in RUN_DIRECTORIES.items():
    print(model)
    for directory in directories:
        print(f'  {directory.name}: exists={directory.exists()}')

In [ ]:
metrics, completion, issues = build_report_metrics(
    ground_truth_path=GROUND_TRUTH_PATH,
    development_pmids_path=DEV_PMIDS_PATH,
    run_directories=RUN_DIRECTORIES,
)

print(f'Evaluated report files: {len(metrics)}')
display(completion.sort_values(['model', 'split']).reset_index(drop=True))
if not issues.empty:
    print(f'Discovery issues: {len(issues)}')
    display(issues.head(20))

## Metric interpretation

- **Context included recall**: proportion of Cochrane-included NCT IDs present anywhere in the final combined context.
- **Ground-truth citation recall**: proportion of Cochrane-included NCT IDs occurring anywhere in the complete generated report, including the Summary and References sections.
- **Grounded citation recall**: proportion of included NCT IDs occurring in the report and also present in the final context.
- **Context-conditioned citation recall**: proportion of the included NCT IDs available in the final context that occur in the report.
- **Summary-only diagnostics**: body-level metrics remain available for diagnosing whether identifiers are used in the narrative rather than only listed in References, but they are not the principal thesis metrics. Identifier overlap establishes traceability, not semantic claim support.

In [ ]:
macro = macro_summary(metrics)
macro_columns = [
    'model', 'split', 'reports',
    'report_included_recall',
    'grounded_report_included_recall',
    'conditional_report_included_recall',
]
display(
    macro[macro_columns]
    .sort_values(['split', 'model'])
    .style.format(precision=3, na_rep='--')
)

### Micro-averaged results

Macro averages give equal weight to every review. Micro averages aggregate matched and reference NCT IDs first, giving more influence to reviews with more included trials.

In [ ]:
micro = micro_summary(metrics)
micro_columns = [
    'model', 'split', 'reports',
    'micro_report_included_recall',
    'micro_grounded_report_included_recall',
    'micro_conditional_report_included_recall',
]
display(
    micro[micro_columns].sort_values(['split', 'model'])
    .style.format(precision=3, na_rep='--')
)

## Report-level citation coverage

The difference between context recall and grounded report recall measures the included-trial coverage lost during report generation. Conditional recall isolates generation behaviour from upstream context availability.

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plot_data = (
    macro.loc[macro['split'] == 'all']
    .set_index('model')[
        [
            'report_included_recall',
            'grounded_report_included_recall',
            'conditional_report_included_recall',
        ]
    ]
    .rename(
        columns={
            'report_included_recall': 'GT citation recall',
            'grounded_report_included_recall': 'Grounded citation recall',
            'conditional_report_included_recall': 'Context-conditioned recall',
        }
    )
)
ax = plot_data.plot(
    kind='bar',
    figsize=(9, 5),
    color=['#246a73', '#d08c32', '#6b8f71'],
    width=0.75,
)
ax.set_title('Report-level citation coverage')
ax.set_xlabel('')
ax.set_ylabel('Macro recall / rate')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=0)
ax.legend(frameon=True, loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'report_level_citation_coverage.pdf', bbox_inches='tight')
plt.show()

## Citation categories

Citations are classified using the Cochrane review sections with the precedence included, ongoing, awaiting assessment, excluded, and uncategorized. Uncategorized citations are not automatically errors: they may represent relevant registry records that were not listed in the review.

In [ ]:
status_summary = citation_status_summary(metrics)
status_columns = [
    'report_included_citation_share',
    'report_ongoing_citation_share',
    'report_awaiting_citation_share',
    'report_excluded_citation_share',
    'report_uncategorized_citation_share',
]
display(
    status_summary.sort_values(['split', 'model'])
    .style.format(precision=3, na_rep='--')
)

status_plot = (
    status_summary.loc[status_summary['split'] == 'all']
    .set_index('model')[status_columns]
    .rename(
        columns={
            'report_included_citation_share': 'Included',
            'report_ongoing_citation_share': 'Ongoing',
            'report_awaiting_citation_share': 'Awaiting',
            'report_excluded_citation_share': 'Excluded',
            'report_uncategorized_citation_share': 'Uncategorized',
        }
    )
)
ax = status_plot.plot(
    kind='barh', stacked=True, figsize=(9, 4),
    color=['#246a73', '#d08c32', '#d8b85a', '#b64e4e', '#9aa0a6'],
)
ax.set_title('Cochrane status of unique NCT citations in complete reports')
ax.set_xlabel('Share of unique citations aggregated across reviews')
ax.set_ylabel('')
ax.set_xlim(0, 1)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'citation_status_composition.pdf', bbox_inches='tight')
plt.show()

## Format and reference-list diagnostics

In [ ]:
format_summary = (
    metrics.groupby(['model', 'split'])
    .agg(
        reports=('pmid', 'count'),
        mean_summary_words=('summary_word_count', 'mean'),
        mean_paragraphs=('summary_paragraph_count', 'mean'),
        paragraph_compliance=('paragraph_requirement_met', 'mean'),
        complete_format_compliance=('format_compliant', 'mean'),
        body_reference_list_coverage=('body_reference_list_coverage', 'mean'),
        unused_reference_list_rate=('unused_reference_list_rate', 'mean'),
        reports_without_body_citations=('body_unique_citation_count', lambda s: int((s == 0).sum())),
        reports_without_context=('context_present', lambda s: int((~s).sum())),
    )
    .reset_index()
)
display(format_summary.style.format(precision=3, na_rep='--'))

problem_columns = [
    'model', 'split', 'pmid', 'summary_paragraph_count',
    'has_summary_heading', 'has_references_heading',
    'summary_has_bullets', 'body_unique_citation_count',
    'body_missing_from_reference_count', 'unused_reference_count',
]
format_problems = metrics.loc[
    (~metrics['format_compliant'])
    | (metrics['body_missing_from_reference_count'] > 0)
    | (metrics['unused_reference_count'] > 0),
    problem_columns,
]
display(format_problems.head(30))

## Paired model comparison

Only PMIDs currently available for both models are included. This table reports descriptive paired mean differences; uncertainty intervals can be added after both final runs are complete.

In [ ]:
paired_metrics = [
    'report_included_recall',
    'grounded_report_included_recall',
    'conditional_report_included_recall',
]
paired_rows = []
models = list(RUN_DIRECTORIES)
if len(models) == 2:
    for metric in paired_metrics:
        wide = metrics.pivot(index='pmid', columns='model', values=metric).dropna()
        if set(models).issubset(wide.columns):
            difference = wide[models[1]] - wide[models[0]]
            paired_rows.append({
                'metric': metric,
                'paired_reviews': len(difference),
                f'{models[1]} minus {models[0]}': difference.mean(),
                f'{models[1]} better': int((difference > 0).sum()),
                'equal': int((difference == 0).sum()),
                f'{models[0]} better': int((difference < 0).sum()),
            })
paired_comparison = pd.DataFrame(paired_rows)
display(paired_comparison.style.format(precision=3, na_rep='--'))

## Export reproducible tables

Rerunning this cell replaces the derived CSV tables with results from the reports currently available. It does not modify the original reports or contexts.

In [ ]:
exported = export_tables(
    metrics=metrics,
    completion=completion,
    issues=issues,
    output_directory=RESULTS_DIR,
)
for name, path in exported.items():
    print(f'{name}: {path}')